# Week 54, Day 3: DP on trees and graphs

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w054d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
children = {0: [1, 2], 1: [3, 4], 2: [5], 3: [], 4: [], 5: []}
order, stack = [], [0]
while stack:
    node = stack.pop()
    order.append(node)
    stack.extend(children[node])
print(order)

size = {}
for node in reversed(order):
    size[node] = 1 + sum(size[c] for c in children[node])
print(size)

In [ ]:
def rooted(n, edges, root=0):
    """Parent of each node and an order with parents before children."""
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    parent, order, stack = [None] * n, [], [root]
    seen = [False] * n
    seen[root] = True
    while stack:
        u = stack.pop()
        order.append(u)
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                stack.append(v)
    return parent, order

parent, order = rooted(6, [(0, 1), (0, 2), (1, 3), (1, 4), (2, 5)])
print(parent, order)

In [ ]:
def rooted(n, edges, root=0):
    """Parent of each node and an order with parents before children."""
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    parent, order, stack = [None] * n, [], [root]
    seen = [False] * n
    seen[root] = True
    while stack:
        u = stack.pop()
        order.append(u)
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                stack.append(v)
    return parent, order


def best_party(fun, edges):
    n = len(fun)
    parent, order = rooted(n, edges)
    take, skip = list(fun), [0] * n
    for v in reversed(order):
        p = parent[v]
        if p is not None:
            take[p] += skip[v]
            skip[p] += max(take[v], skip[v])
    return max(take[0], skip[0])

print(best_party([5, 3, 6, 2, 4, 1], [(0, 1), (0, 2), (1, 3), (1, 4), (2, 5)]))

In [ ]:
def rooted(n, edges, root=0):
    """Parent of each node and an order with parents before children."""
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    parent, order, stack = [None] * n, [], [root]
    seen = [False] * n
    seen[root] = True
    while stack:
        u = stack.pop()
        order.append(u)
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                stack.append(v)
    return parent, order


def all_distances(n, edges):
    parent, order = rooted(n, edges)
    size, down = [1] * n, [0] * n
    for v in reversed(order):
        p = parent[v]
        if p is not None:
            size[p] += size[v]
            down[p] += down[v] + size[v]
    ans = [0] * n
    ans[order[0]] = down[order[0]]
    for v in order[1:]:
        ans[v] = ans[parent[v]] - size[v] + (n - size[v])
    return ans

print(all_distances(6, [(0, 1), (0, 2), (2, 3), (2, 4), (2, 5)]))

In [ ]:
from collections import deque

def earliest_finish(durations, before):
    """before: (a, b) pairs, meaning a must finish before b starts."""
    succ = {t: [] for t in durations}
    need = {t: 0 for t in durations}
    for a, b in before:
        succ[a].append(b)
        need[b] += 1
    start = {t: 0 for t in durations}
    queue = deque(t for t in durations if need[t] == 0)
    finish = {}
    while queue:
        t = queue.popleft()
        finish[t] = start[t] + durations[t]
        for nxt in succ[t]:
            start[nxt] = max(start[nxt], finish[t])
            need[nxt] -= 1
            if need[nxt] == 0:
                queue.append(nxt)
    return finish

tasks = {"dig": 3, "pour": 2, "frame": 4, "wire": 2, "roof": 3, "paint": 1}
deps = [("dig", "pour"), ("pour", "frame"), ("frame", "wire"), ("frame", "roof"), ("wire", "paint"), ("roof", "paint")]
print(earliest_finish(tasks, deps))

> **Key idea.** On a tree, walk from the root with a stack (skipping the parent) to get an order with parents first, then process it backwards so children come first: no deep recursion. Take-or-skip problems keep two values per node; path problems combine the best two children's values at each node; rerooting gets every node's answer in O(n) with a down pass and an up pass that adjusts the parent's answer for the child. On a DAG, a topological order is the fill order: longest and shortest paths, path counts, and the critical path of a project, whose tasks have no slack.

### Exercise: The office party

Employees are numbered 0 to n − 1, with `fun[i]` each; `edges` are the manager links of the org chart (a tree, root 0). Write `best_party(fun, edges)`: `(total, guests)`, the most fun with no employee attending alongside their direct manager, and a sorted list of guests. After computing take and skip, walk **down** from the root: a node attends if its parent doesn't and take ≥ skip.

In [ ]:
def best_party(fun, edges):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(fun, edges):\n    n = len(fun)\n    best = 0\n    for picks in itertools.product([0, 1], repeat=n):\n        if all(not (picks[u] and picks[v]) for u, v in edges):\n            best = max(best, sum(f for f, p in zip(fun, picks) if p))\n    return best\ndef random_tree(rng, n):\n    return [(rng.randrange(i), i) for i in range(1, n)]\ndef test_party():\n    \"independent sets\"\n    total, guests = best_party([5, 3, 6, 2, 4, 1], [(0, 1), (0, 2), (1, 3), (1, 4), (2, 5)])\n    assert total == 12 and guests == [0, 3, 4, 5]\n    assert best_party([7], []) == (7, [0])\n    rng = random.Random(1)\n    for _ in range(100):\n        n = rng.randint(1, 10)\n        fun = [rng.randint(0, 9) for _ in range(n)]\n        edges = random_tree(rng, n)\n        rng.shuffle(edges)\n        total, guests = best_party(fun, edges)\n        chosen = set(guests)\n        assert total == brute(fun, edges) == sum(fun[g] for g in guests)\n        assert all(not (u in chosen and v in chosen) for u, v in edges)\n    n = 100_000\n    stick = [(i, i + 1) for i in range(n - 1)]\n    assert best_party([1] * n, stick)[0] == n // 2")

### Exercise: The longest trail

A park's trails form a tree: `trails` are `(a, b, metres)` between junctions 0 to n − 1. Write `longest_trail(n, trails)`: the length of the longest path between any two junctions, by combining the two largest `down` values at each node. Use no recursion: the checks include a 100,000-junction path.

In [ ]:
def longest_trail(n, trails):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(n, trails):\n    adj = [[] for _ in range(n)]\n    for a, b, w in trails:\n        adj[a].append((b, w))\n        adj[b].append((a, w))\n    best = 0\n    for s in range(n):\n        dist, stack = {s: 0}, [s]\n        while stack:\n            u = stack.pop()\n            for v, w in adj[u]:\n                if v not in dist:\n                    dist[v] = dist[u] + w\n                    stack.append(v)\n        best = max(best, max(dist.values()))\n    return best\ndef test_diameter():\n    \"diameter\"\n    assert longest_trail(1, []) == 0 and longest_trail(2, [(0, 1, 5)]) == 5\n    assert longest_trail(5, [(0, 1, 3), (1, 2, 4), (1, 3, 2), (3, 4, 6)]) == 12\n    rng = random.Random(2)\n    for _ in range(100):\n        n = rng.randint(1, 12)\n        trails = [(rng.randrange(i), i, rng.randint(1, 9)) for i in range(1, n)]\n        assert longest_trail(n, trails) == brute(n, trails)\n    n = 100_000\n    assert longest_trail(n, [(i, i + 1, 2) for i in range(n - 1)]) == 2 * (n - 1)")

### Exercise: The best place for the depot

Towns 0 to n − 1 are joined by roads of length 1 forming a tree. Write `total_distances(n, roads)`: a list giving, for each town, the sum of its distances to every other town, by rerooting in O(n). Then `best_depot(n, roads)`: the town with the smallest total (lowest number on a tie).

In [ ]:
def total_distances(n, roads):
    ___

def best_depot(n, roads):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import deque\ndef brute(n, roads):\n    adj = [[] for _ in range(n)]\n    for a, b in roads:\n        adj[a].append(b)\n        adj[b].append(a)\n    out = []\n    for s in range(n):\n        dist = {s: 0}\n        q = deque([s])\n        while q:\n            u = q.popleft()\n            for v in adj[u]:\n                if v not in dist:\n                    dist[v] = dist[u] + 1\n                    q.append(v)\n        out.append(sum(dist.values()))\n    return out\ndef test_reroot():\n    \"rerooting\"\n    assert total_distances(6, [(0, 1), (0, 2), (2, 3), (2, 4), (2, 5)]) == [8, 12, 6, 10, 10, 10]\n    assert best_depot(6, [(0, 1), (0, 2), (2, 3), (2, 4), (2, 5)]) == 2\n    assert total_distances(1, []) == [0]\n    rng = random.Random(3)\n    for _ in range(100):\n        n = rng.randint(1, 12)\n        roads = [(rng.randrange(i), i) for i in range(1, n)]\n        assert total_distances(n, roads) == brute(n, roads)\n    n = 100_000\n    d = total_distances(n, [(i, i + 1) for i in range(n - 1)])\n    assert d[0] == n * (n - 1) // 2 and best_depot(n, [(i, i + 1) for i in range(n - 1)]) == (n - 1) // 2")

### Exercise: Routes through a DAG

Write `count_paths(n, edges, s, t)`: the number of different directed paths from s to t in a DAG with nodes 0 to n − 1, and `longest_path(n, edges, s, t)`: `(length, path)` for the longest path from s to t by total weight (`edges` are `(u, v, w)`; ignore w for counting), or `None` if t can't be reached. Use Kahn's algorithm for the order.

In [ ]:
from collections import deque

def count_paths(n, edges, s, t):
    ___

def longest_path(n, edges, s, t):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef all_paths(n, edges, s, t):\n    out = []\n    adj = [[] for _ in range(n)]\n    for u, v, w in edges:\n        adj[u].append((v, w))\n    def go(u, path, length):\n        if u == t:\n            out.append((length, path))\n        for v, w in adj[u]:\n            go(v, path + [v], length + w)\n    go(s, [s], 0)\n    return out\ndef test_dag():\n    \"DAG paths\"\n    edges = [(0, 1, 2), (0, 2, 4), (1, 2, 1), (1, 3, 7), (2, 3, 3), (3, 4, 1)]\n    assert count_paths(5, edges, 0, 4) == 3 and longest_path(5, edges, 0, 4) == (10, [0, 1, 3, 4])\n    assert count_paths(5, edges, 4, 0) == 0 and longest_path(5, edges, 4, 0) is None\n    assert longest_path(5, edges, 2, 2) == (0, [2]) and count_paths(5, edges, 2, 2) == 1\n    rng = random.Random(4)\n    for _ in range(100):\n        n = rng.randint(1, 8)\n        edges = [(u, v, rng.randint(-5, 9)) for u in range(n) for v in range(u + 1, n) if rng.random() < 0.4]\n        perm = list(range(n))\n        rng.shuffle(perm)\n        edges = [(perm[u], perm[v], w) for u, v, w in edges]\n        s, t = rng.randrange(n), rng.randrange(n)\n        paths = all_paths(n, edges, s, t)\n        assert count_paths(n, edges, s, t) == len(paths)\n        got = longest_path(n, edges, s, t)\n        if not paths:\n            assert got is None\n        else:\n            assert got[0] == max(p[0] for p in paths) and got in paths")

### Exercise: The critical path

A project has tasks with durations and `before` pairs `(a, b)`: a must finish before b starts. Write `schedule(durations, before)`: `(length, critical)`, the earliest the whole project can finish, and the sorted list of **critical** tasks (those with no slack). Compute each task's earliest start going forwards in topological order, and its latest start going **backwards** (the latest it can start without delaying the project); critical tasks have earliest = latest. Raise `ValueError` if the dependencies have a cycle.

In [ ]:
from collections import deque

def schedule(durations, before):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef test_schedule():\n    \"critical paths\"\n    tasks = {\"dig\": 3, \"pour\": 2, \"frame\": 4, \"wire\": 2, \"roof\": 3, \"paint\": 1}\n    deps = [(\"dig\", \"pour\"), (\"pour\", \"frame\"), (\"frame\", \"wire\"), (\"frame\", \"roof\"), (\"wire\", \"paint\"), (\"roof\", \"paint\")]\n    assert schedule(tasks, deps) == (13, [\"dig\", \"frame\", \"paint\", \"pour\", \"roof\"])\n    assert schedule({\"a\": 5}, []) == (5, [\"a\"]) and schedule({}, []) == (0, [])\n    assert schedule({\"a\": 2, \"b\": 3}, []) == (3, [\"b\"])\n    try:\n        schedule({\"a\": 1, \"b\": 1}, [(\"a\", \"b\"), (\"b\", \"a\")])\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(5)\n    for _ in range(80):\n        n = rng.randint(1, 7)\n        names = [f\"t{i}\" for i in range(n)]\n        durs = {t: rng.randint(1, 6) for t in names}\n        deps = [(names[i], names[j]) for i in range(n) for j in range(i + 1, n) if rng.random() < 0.3]\n        length, critical = schedule(durs, deps)\n        succ = {t: [b for a, b in deps if a == t] for t in names}\n        def longest_from(t):\n            return durs[t] + max([longest_from(s) for s in succ[t]], default=0)\n        assert length == max(longest_from(t) for t in names)\n        for t in names:\n            delayed = dict(durs)\n            delayed[t] += 1\n            longer = schedule(delayed, deps)[0] > length\n            assert longer == (t in critical), (t, durs, deps)")

### Bug hunt: Parents before children

The walk lists every node after its parent, so a tree DP must read that list **backwards**. This loop runs forwards, adding each node's size to its parent before the node's own children have been counted, and prints `{0: 3, 1: 2, 2: 1, 3: 1}`. Process the order in reverse so it prints `{0: 4, 1: 2, 2: 1, 3: 1}`.

In [ ]:
adj = {0: [1, 2], 1: [0, 3], 2: [0], 3: [1]}
order, stack, parent = [], [(0, None)], {}
while stack:
    node, p = stack.pop()
    order.append(node)
    parent[node] = p
    for nxt in adj[node]:
        if nxt != p:
            stack.append((nxt, node))
size = {v: 1 for v in adj}
for v in order:
    if parent[v] is not None:
        size[parent[v]] += size[v]
print(dict(sorted(size.items())))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"children first\"\n    expect_output(\"{0: 4, 1: 2, 2: 1, 3: 1}\")")

### Bug hunt: Manager and employee, both invited

In the party DP, if v attends, its children must **not**: take(parent) should add skip(child). This adds take(child) instead, so a manager and their report both count, and it prints `21`. Fix the line so it prints `12`.

In [ ]:
fun = [5, 3, 6, 2, 4, 1]
parent = [None, 0, 0, 1, 1, 2]
order = [0, 2, 5, 1, 4, 3]
take, skip = list(fun), [0] * 6
for v in reversed(order):
    p = parent[v]
    if p is not None:
        take[p] += take[v]
        skip[p] += max(take[v], skip[v])
print(max(take[0], skip[0]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"independent\"\n    expect_output(\"12\")")

### Bug hunt: Not in topological order

This counts paths in a DAG by visiting nodes in number order, but node 3 points to node 1, so node 1's count is passed on before its own predecessors have finished adding to it. It prints `1` instead of `2`. Visit the nodes in topological order (here `[0, 3, 1, 2]`).

In [ ]:
edges = {0: [3, 1], 3: [1], 1: [2], 2: []}
ways = {v: 0 for v in edges}
ways[0] = 1
for u in sorted(edges):
    for v in edges[u]:
        ways[v] += ways[u]
print(ways[2])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"topological order\"\n    expect_output(\"2\")")

### Puzzle: Rerooting

Arrange the two passes of the sum-of-distances rerooting, given `parent` and `order` (parents first). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the best party for a path of three people with fun 1, 5, 1 (0 manages 1, 1 manages 2)
q = ___   # the subtree size of the root in a tree of 7 nodes
r = ___   # True or False: longest path is easy on any directed graph
s = ___   # the number of paths from 0 to 3 in edges 0→1, 0→2, 1→3, 2→3, 0→3
t = ___   # the diameter (in edges) of a star with one centre and 5 leaves

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 5 and q == 7 and r is False, \"p, q and r\"\n    assert s == 3 and t == 2, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: watching every junction

A security firm puts cameras at junctions of a tree-shaped network of corridors (junctions 0 to n − 1). A camera watches its own junction and every junction **next to it**. Write `fewest_cameras(n, corridors)`: the fewest cameras so every junction is watched, and `camera_plan(n, corridors)`: a sorted list of junctions for one such plan. (This is a minimum dominating set, NP-hard on general graphs and a tree DP here.)

Keep three values per node v, for v's subtree:

- **has(v)**: v has a camera (its children may be in any state; each child is watched by v);
- **watched(v)**: no camera at v, but some child has one, and every node in the subtree is watched;
- **needs(v)**: no camera at v and no child has one; everything below is watched, but v itself still needs its **parent** to have a camera.

has(v) = 1 + sum of min(has(c), watched(c), needs(c)); needs(v) = sum of watched(c); watched(v) = sum of min(has(c), watched(c)) plus the cheapest way to force at least one child to have a camera. The answer is min(has(root), watched(root)). Use no recursion.

In [ ]:
def fewest_cameras(n, corridors):
    ___

def camera_plan(n, corridors):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef watches(n, edges, cams):\n    seen = set(cams)\n    for a, b in edges:\n        if a in cams:\n            seen.add(b)\n        if b in cams:\n            seen.add(a)\n    return len(seen) == n\ndef brute(n, edges):\n    for k in range(1, n + 1):\n        for cams in itertools.combinations(range(n), k):\n            if watches(n, edges, set(cams)):\n                return k\n    return 0\ndef test_cameras():\n    \"dominating sets\"\n    assert fewest_cameras(1, []) == 1 and camera_plan(1, []) == [0]\n    assert fewest_cameras(5, [(0, 1), (1, 2), (2, 3), (3, 4)]) == 2\n    assert fewest_cameras(4, [(0, 1), (0, 2), (0, 3)]) == 1 and camera_plan(4, [(0, 1), (0, 2), (0, 3)]) == [0]\n    rng = random.Random(6)\n    for _ in range(100):\n        n = rng.randint(1, 10)\n        edges = [(rng.randrange(i), i) for i in range(1, n)]\n        plan = camera_plan(n, edges)\n        assert fewest_cameras(n, edges) == brute(n, edges) == len(plan)\n        assert watches(n, edges, set(plan)) and plan == sorted(set(plan))\n    n = 100_000\n    stick = [(i, i + 1) for i in range(n - 1)]\n    assert fewest_cameras(n, stick) == (n + 2) // 3")

## Warm-up

The loop lists every node after its parent (a depth-first order using a stack). Reading that list **backwards** puts every child before its parent, which is exactly the fill order a tree DP needs: by the time a node is processed, all its children are done.

## Children before parents

A rooted tree's DP defines a value per node from the values of its **children**: the size of its subtree, the best answer within its subtree, and so on. Recursion expresses that naturally, but a tree can be a long path (a "stick" of 100,000 nodes), far past the recursion limit. The standard fix:

1. Build the tree as an adjacency list from the edges, then walk it from the root with a stack, recording each node's **parent** and the **order** nodes were reached.
2. Process the nodes in **reverse** order: every child is then handled before its parent.

The walk from the root must skip the node it came from (each undirected edge appears in both nodes' lists). Forgetting the parent check makes the walk go back up the tree.

## Take it or leave it

A company wants a party: each employee has a "fun" score, and nobody should attend with their **direct manager** (the org chart is a tree). Maximise the total fun. It's house robber on a tree: for each node keep two values,

- **take(v)** = the best in v's subtree if v attends = fun(v) + sum of skip(c) over children c;
- **skip(v)** = the best if v doesn't = sum of max(take(c), skip(c)) over children.

The answer is max(take(root), skip(root)). Two values per node, children first: O(n). (The general problem, a maximum-weight independent set, is NP-hard on arbitrary graphs; on trees it's this easy. Many hard graph problems become DPs on trees.)

The children push their values into the parent's running totals, which avoids building a list of children at all.

## Combining children

Some answers aren't "a value in this subtree" but "a path through this node". The **diameter** of a weighted tree, the longest path between any two nodes, passes through some highest node v and goes down into two different children's subtrees. So keep

**down(v) = the longest path from v down into its subtree**,

and at each node combine the two largest values of down(c) + w(v, c) over its children: that's the longest path whose highest node is v. The diameter is the best over all nodes. It's "best ending here" again, with a tree for the line.

## Rerooting

Some questions want an answer for **every** node as if it were the root: "for each town, the total distance to all the other towns". Running a DP from every root costs O(n²). **Rerooting** gets all n answers in O(n) with two passes:

1. **Down pass** (children first): for a root of 0, compute `size(v)`, the number of nodes in v's subtree, and `down(v)`, the total distance from v to the nodes in its subtree: down(v) = sum of down(c) + size(c) over children (every node under c is one edge further from v).
2. **Up pass** (parents first): the answer at the root is down(0). Moving the root from p to its child c, the size(c) nodes in c's subtree each get one edge closer, and the other n − size(c) nodes one edge further: ans(c) = ans(p) − size(c) + (n − size(c)).

The second pass walks the order **forwards**, so each parent's answer is ready before its children's. The same pattern answers "the furthest node from each node", "the best place to put a depot" and more.

## DAGs in topological order

In a directed acyclic graph, a **topological order** lists every node before all the nodes it points to (Week 50's Kahn's algorithm). That's a fill order for any DP where a node's value depends on its predecessors:

- **Longest path** from a source: longest(v) = max over edges u → v of longest(u) + w. (On general graphs longest path is hard; on DAGs it's this loop. Shortest paths work the same way, even with negative weights.)
- **Counting paths** from s to t: paths(v) = sum of paths(u) over edges u → v, with paths(s) = 1.
- **Project scheduling**: tasks with durations and "must finish before" dependencies. The earliest each task can finish is its duration plus the latest finish among its prerequisites. The project's length is the longest path, and the tasks on it are **critical**: delay any one and the whole project slips. This is the **critical path method**, developed by James Kelley and Morgan Walker in the late 1950s for planning chemical plant work.

## Practice

## Quiz

## Daily challenge

## Before you go

Draw a family tree or an org chart you know, give each person a score, and work out the best party by hand with take and skip. Tomorrow: DP over **sets**, with bitmasks.